In [1]:
import pandas as pd

In [42]:
curve_df = pd.read_hdf('./data/data.h5', key='curve_data')
sample_info = pd.read_hdf('./data/data.h5', key='sample_info')
igi_gene_call = pd.read_hdf('./data/data.h5', key='igi_gene_call')

In [19]:
curve_df.loc[:,'rounded_cq'] = [int(float(x)) if x != 'Undetermined' else x for x in curve_df.cq]
fn_threshold_df = (curve_df
                    .loc[curve_df.cycle_no == curve_df.rounded_cq, ['curve_idx','cycle_no','cq', 'Fn']]
                    .rename(columns={'Fn':'fn_threshold'}))
fn_threshold_df.loc[:,'fn_low'] = fn_threshold_df.fn_threshold - 0.05*fn_threshold_df.fn_threshold
fn_threshold_df.loc[:,'fn_high'] = fn_threshold_df.fn_threshold + 0.05*fn_threshold_df.fn_threshold

In [56]:
std_df = (curve_df
    .loc[(curve_df.cycle_no <= curve_df.baseline_end)]
    .groupby(['pcr_plate','target'])
    .Fn.std().reset_index())

std_df.loc[:,'fn_threshold'] = std_df.Fn*10
std_df.loc[:,'fn_low1'] = std_df.fn_threshold - 0.05*std_df.fn_threshold
std_df.loc[:,'fn_high1'] = std_df.fn_threshold + 0.05*std_df.fn_threshold
std_df = std_df.drop('Fn', axis = 1)

In [ ]:
threshold_df = curve_df[['curve_idx','cq','threshold']].drop_duplicates()
threshold_df.loc[:,'drn_low'] = threshold_df.threshold - 0.05*threshold_df.threshold
threshold_df.loc[:,'drn_high'] = threshold_df.threshold + 0.05*threshold_df.threshold

In [10]:
max_cycle_df = curve_df.groupby(['curve_idx']).cycle_no.agg(max).reset_index()
luner_ids = max_cycle_df[max_cycle_df.cycle_no == 45].curve_idx.unique()
thermo_ids = max_cycle_df[max_cycle_df.cycle_no == 40].curve_idx.unique()

In [62]:
join_df = (curve_df
          .merge(threshold_df[['curve_idx','drn_low','drn_high']], how = 'inner', on = 'curve_idx')
          .merge(fn_threshold_df[['curve_idx','fn_threshold','fn_low','fn_high']], how = 'inner', on = 'curve_idx')
          .merge(std_df, how='inner', on = ['pcr_plate', 'target']))

In [63]:
amb_df = (join_df
            .loc[((join_df.curve_idx.isin(luner_ids)) & (join_df.cycle_no >= 40)) | 
                        ((join_df.curve_idx.isin(thermo_ids)) & (join_df.cycle_no >= 35))]
           .loc[(join_df.drn <= join_df.drn_high) & (join_df.drn >= join_df.drn_low)])
amb_df.curve_idx.nunique()

653

In [64]:
amb_df = (join_df
            .loc[((join_df.curve_idx.isin(luner_ids)) & (join_df.cycle_no >= 40)) | 
                        ((join_df.curve_idx.isin(thermo_ids)) & (join_df.cycle_no >= 35))]
           .loc[(join_df.Fn <= join_df.fn_high1) & (join_df.Fn >= join_df.fn_low1)])
amb_df.curve_idx.nunique()

1375

In [65]:
amb_df[['curve_idx','cq','threshold','cycle_no','Fn','drn','drn_low','drn_high','fn_low','fn_high']]

,curve_idx,cq,threshold,cycle_no,Fn,drn,drn_low,drn_high,fn_low,fn_high
37,23616,26.07906497442115,4189.194,38,245175.600,87377.637664,3979.7343,4398.6537,154587.344,170859.696
38,23616,26.07906497442115,4189.194,39,254079.900,96350.501069,3979.7343,4398.6537,154587.344,170859.696
39,23616,26.07906497442115,4189.194,40,261890.660,104229.801974,3979.7343,4398.6537,154587.344,170859.696
74,23627,23.57513092064245,4189.194,35,247405.310,20829.281771,3979.7343,4398.6537,219642.356,242762.604
75,23627,23.57513092064245,4189.194,36,249982.470,23503.192708,3979.7343,4398.6537,219642.356,242762.604
...,...,...,...,...,...,...,...,...,...,...
1194835,128498,39.721298219723636,18212.000,36,122166.740,10401.619216,17301.4000,19122.6000,122710.360,135627.240
1195074,128494,39.95485111707075,18212.000,35,112587.490,5158.835793,17301.4000,19122.6000,117476.202,129842.118
1195075,128494,39.95485111707075,18212.000,36,114994.950,7425.602628,17301.4000,19122.6000,117476.202,129842.118
1195076,128494,39.95485111707075,18212.000,37,117643.336,9933.291338,17301.4000,19122.6000,117476.202,129842.118


In [67]:
join_df.loc[join_df.curve_idx == 23616,['curve_idx','pcr_plate','target','well_position','cq','threshold','cycle_no','Fn','drn','drn_low','drn_high','fn_low','fn_high', 'fn_low1','fn_high1']]

,curve_idx,pcr_plate,target,well_position,cq,threshold,cycle_no,Fn,drn,drn_low,drn_high,fn_low,fn_high,fn_low1,fn_high1
0,23616,AC00DB1I,MS2,A1,26.07906497442115,4189.194,1,160733.60,399.254194,3979.7343,4398.6537,154587.344,170859.696,240075.261034,265346.341143
1,23616,AC00DB1I,MS2,A1,26.07906497442115,4189.194,2,160684.52,418.726974,3979.7343,4398.6537,154587.344,170859.696,240075.261034,265346.341143
2,23616,AC00DB1I,MS2,A1,26.07906497442115,4189.194,3,160373.90,176.668503,3979.7343,4398.6537,154587.344,170859.696,240075.261034,265346.341143
3,23616,AC00DB1I,MS2,A1,26.07906497442115,4189.194,4,160091.80,-36.889967,3979.7343,4398.6537,154587.344,170859.696,240075.261034,265346.341143
4,23616,AC00DB1I,MS2,A1,26.07906497442115,4189.194,5,160030.38,-29.760937,3979.7343,4398.6537,154587.344,170859.696,240075.261034,265346.341143
5,23616,AC00DB1I,MS2,A1,26.07906497442115,4189.194,6,160304.42,312.836842,3979.7343,4398.6537,154587.344,170859.696,240075.261034,265346.341143
6,23616,AC00DB1I,MS2,A1,26.07906497442115,4189.194,7,160208.56,285.528372,3979.7343,4398.6537,154587.344,170859.696,240075.261034,265346.341143
7,23616,AC00DB1I,MS2,A1,26.07906497442115,4189.194,8,159773.53,-80.951974,3979.7343,4398.6537,154587.344,170859.696,240075.261034,265346.341143
8,23616,AC00DB1I,MS2,A1,26.07906497442115,4189.194,9,159597.06,-188.869819,3979.7343,4398.6537,154587.344,170859.696,240075.261034,265346.341143
9,23616,AC00DB1I,MS2,A1,26.07906497442115,4189.194,10,159549.60,-167.787664,3979.7343,4398.6537,154587.344,170859.696,240075.261034,265346.341143


In [43]:
comprehensive_df = (amb_df
                    .merge(sample_info, how='inner', on=['well_position','pcr_plate'])
                    .merge(igi_gene_call, how='inner', on=['pcr_plate','sample_id','target']))

In [44]:
(comprehensive_df
    .groupby(['sample_type','igi_call','target'])
    .curve_idx
    .nunique())

sample_type                                 igi_call  target
Buffer Negative Control (Extraction)        Negative  E gene      6
                                                      N gene      1
                                                      ORF1ab      1
                                                      RnaseP      1
                                                      S gene      1
                                            Positive  MS2         1
Clinical Sample                             Negative  E gene     84
                                                      MS2        21
                                                      N gene    110
                                                      ORF1ab     28
                                                      RnaseP     44
                                                      S gene     61
                                            Positive  MS2        58
                                                      N

In [76]:
import numpy as np
curve_df.loc[curve_df.pcr_plate == 'AC00DB1I'].groupby('target').agg(threshold = ('drn', lambda x: np.std(x)))

,threshold
target,
MS2,64001.557309
N gene,73400.615725
ORF1ab,100427.488225
S gene,87461.373275
